# 07 — Evaluation
Measures the fine-tuned model across five dimensions:
1. **Syntax pass rate** — generated ARO code passes `aro check`
2. **Debugging accuracy** — fixed code passes, buggy code fails
3. **Token overlap** — Q&A / explanation quality vs reference answers
4. **Hallucination rate** — model invents non-existent ARO verbs
5. **FIM accuracy** — fill-in-the-middle exact match

Also compares the fine-tuned model against the base model on the same prompts.

**Input:**  `../data/05_dataset/test.jsonl`
            `../data/02_knowledge/knowledge.json`
            `../models/qwen25-coder-7b-aro/lora-adapter/`
**Output:** `../data/07_eval/report.json`  +  printed summary

In [ ]:
import json
import re
import subprocess
import tempfile
import platform
from pathlib import Path
from collections import defaultdict, Counter

try:
    SCRIPT_DIR = Path(__file__).parent.resolve()
except NameError:
    SCRIPT_DIR = Path('.').resolve()

DATA_DIR    = SCRIPT_DIR / '../data/05_dataset'
KB_DIR      = SCRIPT_DIR / '../data/02_knowledge'
ADAPTER_DIR = SCRIPT_DIR / '../models/qwen25-coder-7b-aro/lora-adapter'
EVAL_DIR    = SCRIPT_DIR / '../data/07_eval'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL  = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MAX_EVAL    = 200   # number of test samples to evaluate (set to None for all)
MAX_TOKENS  = 512

## Load test set + knowledge base

In [ ]:
test_data = []
with open(DATA_DIR / 'test.jsonl') as f:
    for line in f:
        if line.strip():
            test_data.append(json.loads(line))

if MAX_EVAL:
    import random
    random.seed(0)
    test_data = random.sample(test_data, min(MAX_EVAL, len(test_data)))

print(f'Test samples: {len(test_data)}')
print('Distribution:', dict(Counter(s['task_type'] for s in test_data)))

# Load known verbs for hallucination check
with open(KB_DIR / 'knowledge.json') as f:
    kb = json.load(f)

known_verbs = set()
for action in kb['actions']:
    for v in action['verbs']:
        known_verbs.add(v.lower())

print(f'Known ARO verbs: {len(known_verbs)}')

## Load models

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

device = (
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Device: {device}')

# Load base tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Fine-tuned model
print('Loading fine-tuned model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map=device, trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
ft_model.eval()

# Base model (for comparison) — share weights, just swap adapter off
print('Loading base model for comparison...')
base_only = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map=device, trust_remote_code=True,
)
base_only.eval()

print('Models ready.')

## Generation + validation helpers

In [ ]:
def build_prompt(sample):
    """Reconstruct the inference prompt (system + user turns only)."""
    msgs = sample['messages'][:2]  # system + user
    text = ''
    for m in msgs:
        text += f'<|im_start|>{m["role"]}\n{m["content"]}<|im_end|>\n'
    text += '<|im_start|>assistant\n'
    return text

def generate(model, prompt, max_tokens=MAX_TOKENS):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]

def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except Exception:
        return None

def token_overlap(generated, reference):
    """F1-style token overlap between generated and reference text."""
    gen_tokens = set(generated.lower().split())
    ref_tokens = set(reference.lower().split())
    if not ref_tokens:
        return 0.0
    precision = len(gen_tokens & ref_tokens) / max(1, len(gen_tokens))
    recall    = len(gen_tokens & ref_tokens) / max(1, len(ref_tokens))
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def hallucination_score(text):
    """Fraction of verbs in generated ARO code that are NOT in known_verbs."""
    blocks = extract_aro_blocks(text)
    if not blocks:
        return None
    verbs_found = re.findall(r'^\s+([A-Z][a-z]+)', '\n'.join(blocks), re.MULTILINE)
    if not verbs_found:
        return None
    unknown = [v for v in verbs_found if v.lower() not in known_verbs]
    return len(unknown) / len(verbs_found)

## Run evaluation

In [ ]:
results = defaultdict(lambda: {
    'total': 0, 'ft_syntax': 0, 'base_syntax': 0,
    'ft_overlap': [], 'base_overlap': [],
    'ft_hallucination': [], 'base_hallucination': [],
    'ft_fim_exact': 0, 'base_fim_exact': 0,
    'debug_fix_pass': 0, 'debug_buggy_fail': 0,
})

detail_rows = []

for i, sample in enumerate(test_data):
    task    = sample['task_type']
    ref     = sample['messages'][2]['content']
    prompt  = build_prompt(sample)

    try:
        ft_out   = generate(ft_model, prompt)
        base_out = generate(base_only, prompt)
    except Exception as e:
        print(f'  ! generate error at {i}: {e}')
        continue

    r = results[task]
    r['total'] += 1

    # --- Syntax pass rate (code tasks) ---
    if task in ('code_generation', 'translation', 'fim'):
        for label, out in [('ft', ft_out), ('base', base_out)]:
            blocks = extract_aro_blocks(out) or [out]  # FIM may have no fences
            code   = '\n\n'.join(blocks)
            passed = aro_check(code)
            if passed is True:
                r[f'{label}_syntax'] += 1

    # --- FIM exact match ---
    if task == 'fim':
        middle = sample.get('middle', '').strip()
        if ft_out.strip() == middle:
            r['ft_fim_exact'] += 1
        if base_out.strip() == middle:
            r['base_fim_exact'] += 1

    # --- Debugging: fix must pass, buggy must fail ---
    if task == 'debugging':
        fix_m = re.search(r'## Fix\s*```aro\n(.*?)```', ft_out, re.DOTALL)
        if fix_m:
            passed = aro_check(fix_m.group(1).strip())
            if passed is True:
                r['debug_fix_pass'] += 1
        buggy_m = re.search(r'## Buggy Code\s*```aro\n(.*?)```', ft_out, re.DOTALL)
        if buggy_m:
            buggy_passed = aro_check(buggy_m.group(1).strip())
            if buggy_passed is False:
                r['debug_buggy_fail'] += 1

    # --- Token overlap (explanation / QA / architecture) ---
    if task in ('code_explanation', 'syntax_qa', 'architecture'):
        r['ft_overlap'].append(token_overlap(ft_out, ref))
        r['base_overlap'].append(token_overlap(base_out, ref))

    # --- Hallucination rate ---
    for label, out in [('ft', ft_out), ('base', base_out)]:
        score = hallucination_score(out)
        if score is not None:
            r[f'{label}_hallucination'].append(score)

    detail_rows.append({
        'task': task, 'ft_output': ft_out[:500], 'base_output': base_out[:500], 'reference': ref[:500],
    })

    if (i + 1) % 20 == 0:
        print(f'  Evaluated {i+1}/{len(test_data)}')

print('Evaluation complete.')

## Print report

In [ ]:
def avg(lst):
    return sum(lst) / len(lst) if lst else 0.0

print('\n' + '=' * 70)
print(f'{"METRIC":<35} {"FINE-TUNED":>12} {"BASE":>12}')
print('=' * 70)

report = {}

for task, r in sorted(results.items()):
    n = r['total']
    if n == 0:
        continue

    report[task] = {}
    print(f'\n  [{task}]  n={n}')

    if r['ft_syntax'] or r['base_syntax']:
        ft_rate   = r['ft_syntax'] / n
        base_rate = r['base_syntax'] / n
        print(f'  {"Syntax pass rate":<33} {ft_rate:>11.1%} {base_rate:>11.1%}')
        report[task]['ft_syntax_pass_rate']   = round(ft_rate, 3)
        report[task]['base_syntax_pass_rate'] = round(base_rate, 3)

    if r['ft_overlap']:
        ft_ov   = avg(r['ft_overlap'])
        base_ov = avg(r['base_overlap'])
        print(f'  {"Token overlap (F1)":<33} {ft_ov:>11.2f} {base_ov:>11.2f}')
        report[task]['ft_token_overlap']   = round(ft_ov, 3)
        report[task]['base_token_overlap'] = round(base_ov, 3)

    if r['ft_hallucination']:
        ft_h   = avg(r['ft_hallucination'])
        base_h = avg(r['base_hallucination'])
        print(f'  {"Hallucination rate (lower=better)":<33} {ft_h:>11.1%} {base_h:>11.1%}')
        report[task]['ft_hallucination_rate']   = round(ft_h, 3)
        report[task]['base_hallucination_rate'] = round(base_h, 3)

    if task == 'fim' and (r['ft_fim_exact'] or r['base_fim_exact']):
        ft_ex   = r['ft_fim_exact'] / n
        base_ex = r['base_fim_exact'] / n
        print(f'  {"FIM exact match":<33} {ft_ex:>11.1%} {base_ex:>11.1%}')
        report[task]['ft_fim_exact_match']   = round(ft_ex, 3)
        report[task]['base_fim_exact_match'] = round(base_ex, 3)

    if task == 'debugging' and r['debug_fix_pass']:
        fix_rate = r['debug_fix_pass'] / n
        print(f'  {"Fix-passes-check rate":<33} {fix_rate:>11.1%}')
        report[task]['debug_fix_pass_rate'] = round(fix_rate, 3)

print('\n' + '=' * 70)

## Save report

In [ ]:
with open(EVAL_DIR / 'report.json', 'w') as f:
    json.dump(report, f, indent=2)

with open(EVAL_DIR / 'details.jsonl', 'w') as f:
    for row in detail_rows:
        f.write(json.dumps(row) + '\n')

print(f'\nReport saved: {EVAL_DIR / "report.json"}')
print(f'Details saved: {EVAL_DIR / "details.jsonl"}')

## Inspect failures

In [ ]:
# Show 5 cases where fine-tuned model produced invalid ARO code
print('=== Sample failures (ft model produced invalid ARO) ===\n')
shown = 0
for row in detail_rows:
    if row['task'] not in ('code_generation', 'translation'):
        continue
    blocks = extract_aro_blocks(row['ft_output'])
    if blocks and aro_check('\n\n'.join(blocks)) is False:
        print(f'Task: {row["task"]}')
        print(f'Generated:\n{row["ft_output"][:400]}\n')
        shown += 1
    if shown >= 5:
        break

if shown == 0:
    print('No failures found in sample set.')